In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
from astropy.wcs import WCS
from astropy import units as u
from reproject import reproject_from_healpix, reproject_to_healpix
from astropy_healpix import HEALPix
from astropy.coordinates import Galactic
from astropy.coordinates import SkyCoord
import healpy
import gc
import seaborn as sns

### Read in STAPS EQ

In [ ]:
hdu     = fits.open('/srv/data/gmims/gmims-hbs/published/staps.i.E.smooth.K.fits')
cube_i = hdu[0].data
hdr     = hdu[0].header

hdu     = fits.open('/srv/data/gmims/gmims-hbs/published/staps.q.E.smooth.K.fits')
cube_q = hdu[0].data

hdu     = fits.open('/srv/data/gmims/gmims-hbs/published/staps.u.E.smooth.K.fits')
cube_u = hdu[0].data

hdr2D = hdr.copy()
print(repr(hdr2D))

hdr2D['NAXIS']  = 2
del hdr2D['NAXIS3']
hdr2D['BUNIT'] = 'K(Tb)'
hdr2D['CUNIT1']   = 'deg' 
hdr2D['CUNIT2']   = 'deg'
hdr2D['COORDSYS'] = 'icrs'


### Read in STAPS frequencies and pick freq channel

In [ ]:
freqs = np.genfromtxt('/srv/data/gmims/gmims-hbs/published/freq.txt')

fpick = 1373

idx = abs(fpick - freqs/1e6).argmin()
print(freqs[idx])

map_i = cube_i[idx]
map_q = cube_q[idx]
map_u = cube_u[idx]


In [ ]:
plt.imshow(map_q, vmin=-0.3, vmax=0.3, cmap='RdBu_r',origin='lower')

### Convert STAPS to Healpix

In [ ]:
map_i_hpx, footprint = reproject_to_healpix((map_i,WCS(hdr2D)),coord_system_out='C',nside=512)
map_q_hpx, footprint = reproject_to_healpix((map_q,WCS(hdr2D)),coord_system_out='C',nside=512)
map_u_hpx, footprint = reproject_to_healpix((map_u,WCS(hdr2D)),coord_system_out='C',nside=512)


In [ ]:
healpy.mollview(map_u_hpx, min=-0.3, max=0.3, cmap='RdBu_r')
healpy.mollview(map_i_hpx, min=0, max=20, cmap='viridis')

### Convert STAPS to Galactic

In [ ]:
map_u_hpx *= -1

map_i_hpx[np.isnan(map_i_hpx)] = healpy.UNSEEN
map_q_hpx[np.isnan(map_q_hpx)] = healpy.UNSEEN
map_u_hpx[np.isnan(map_u_hpx)] = healpy.UNSEEN

r = healpy.Rotator(coord=['C', 'G'])

Igal_hpx,  Qgal_hpx,  Ugal_hpx  = r.rotate_map_alms([map_i_hpx, map_q_hpx, map_u_hpx])
i_rot_pix, q_rot_pix, u_rot_pix = r.rotate_map_pixel([map_i_hpx, map_q_hpx, map_u_hpx])

Igal_hpx[(i_rot_pix < -1e20) | (i_rot_pix > 1e20)] = healpy.UNSEEN
Qgal_hpx[(q_rot_pix < -1e20) | (q_rot_pix > 1e20)] = healpy.UNSEEN
Ugal_hpx[(u_rot_pix < -1e20) | (u_rot_pix > 1e20)] = healpy.UNSEEN

Qgal_hpx = np.array(Qgal_hpx)
Ugal_hpx = np.array(Ugal_hpx)
Igal_hpx = np.array(Igal_hpx)

Qgal_hpx[Qgal_hpx == healpy.UNSEEN] = np.nan
Ugal_hpx[Ugal_hpx == healpy.UNSEEN] = np.nan
Igal_hpx[Igal_hpx == healpy.UNSEEN] = np.nan

map_i_hpx[map_i_hpx == healpy.UNSEEN] = np.nan
map_q_hpx[map_q_hpx == healpy.UNSEEN] = np.nan
map_u_hpx[map_u_hpx == healpy.UNSEEN] = np.nan

Ugal_hpx *= -1
map_u_hpx *= -1


In [ ]:
healpy.mollview(Ugal_hpx,min=-0.3,max=0.3,cmap='RdBu_r')

In [ ]:
healpy.mollview(map_u_hpx, min=-0.3, max=0.3, cmap='RdBu_r')

### Make Galactic

In [ ]:
pixsize = 0.125

nx = int(round((360/pixsize),0))
ny = int(round((180/pixsize),0))

hdr_gal = hdr2D.copy()

hdr_gal['COORDSYS']  = 'galactic'
hdr_gal['NAXIS']   = 2 
hdr_gal['NAXIS1']  = nx 
hdr_gal['NAXIS2']  = ny

hdr_gal['CTYPE1']  = 'GLON-CAR'
hdr_gal['CRPIX1']  = nx/2.+0.5 
hdr_gal['CRVAL1']  = 0          
hdr_gal['CDELT1']  = -pixsize
hdr_gal['CUNIT1']  = 'deg'

hdr_gal['CTYPE2']  = 'GLAT-CAR'
hdr_gal['CRPIX2']  = ny/2.+0.5
hdr_gal['CRVAL2']  = 0
hdr_gal['CDELT2']  = pixsize
hdr_gal['CUNIT2']  = 'deg'

#del hdr_gal['RADESYS']
        
print(repr(hdr_gal))


In [ ]:
hdr2D_fix = hdr_gal.copy()
                                                
hdr2D_fix['CTYPE1']  = 'RA---CAR'                                                            
hdr2D_fix['CTYPE2']  = 'DEC--CAR'                                                                                                             
hdr2D_fix['BUNIT']   = 'K(Tb)   '                                                                                                                      
hdr2D_fix['COORDSYS']= 'icrs'   

print(repr(hdr2D_fix))


### Convert back to plate caree

In [ ]:
Igal, footprint = reproject_from_healpix((Igal_hpx,'galactic'), hdr_gal, nested=False)
Qgal, footprint = reproject_from_healpix((Qgal_hpx,'galactic'), hdr_gal, nested=False)
Ugal, footprint = reproject_from_healpix((Ugal_hpx,'galactic'), hdr_gal, nested=False)

Ieq, footprint = reproject_from_healpix((map_i_hpx,'icrs'), hdr2D_fix, nested=False)
Qeq, footprint = reproject_from_healpix((map_q_hpx,'icrs'), hdr2D_fix, nested=False)
Ueq, footprint = reproject_from_healpix((map_u_hpx,'icrs'), hdr2D_fix, nested=False)

In [ ]:
fits.writeto('/home/aordog/Dropbox/gmims_signs/staps1373_q.gal.fits', Qgal.astype('float32'), hdr_gal, overwrite=True, output_verify='fix')
fits.writeto('/home/aordog/Dropbox/gmims_signs/staps1373_u.gal.fits', Ugal.astype('float32'), hdr_gal, overwrite=True, output_verify='fix')
fits.writeto('/home/aordog/Dropbox/gmims_signs/staps1373_i.gal.fits', Igal.astype('float32'), hdr_gal, overwrite=True, output_verify='fix')

fits.writeto('/home/aordog/Dropbox/gmims_signs/staps1373_q.eq.fits', Qeq.astype('float32'), hdr2D_fix, overwrite=True, output_verify='fix')
fits.writeto('/home/aordog/Dropbox/gmims_signs/staps1373_u.eq.fits', Ueq.astype('float32'), hdr2D_fix, overwrite=True, output_verify='fix')
fits.writeto('/home/aordog/Dropbox/gmims_signs/staps1373_i.eq.fits', Ieq.astype('float32'), hdr2D_fix, overwrite=True, output_verify='fix')


In [ ]:
this = fits.open('/home/aordog/Dropbox/gmims_signs/CHIME_EQ_614.fits')
print(repr(this[0].header))

In [ ]:
plt.imshow(Qeq,vmin=-0.3,vmax=0.3)

### Read in DRAO export package outputs (from rfits -> mapconvrt -> QU_convert -> wfits)

In [ ]:
hdu     = fits.open('/home/aordog/Dropbox/gmims_signs/staps_q_gal_good.fits')
map_new_q = hdu[0].data[0][0]

hdu     = fits.open('/home/aordog/Dropbox/gmims_signs/staps_u_gal_good.fits')
map_new_u = hdu[0].data[0][0]

hdr_new = hdu[0].header
wcs_new = WCS(hdr_new).dropaxis(3).dropaxis(2)
print(wcs_new)

map_new_q_hpx, footprint = reproject_to_healpix((map_new_q,wcs_new),coord_system_out='G',nside=256)
map_new_u_hpx, footprint = reproject_to_healpix((map_new_u,wcs_new),coord_system_out='G',nside=256)

In [ ]:
def make_mollweide_map(gal_hpx_map, vmin, vmax, cbar_ticks,
                       fs=24, cmap='cubehelix', outname='test', title='test',
                       cbarlabel='test'):

    fs_dict = {'xlabel':fs, 'ylabel':fs, 'title':fs+6, 'xtick_label':fs, 
               'ytick_label':fs, 'cbar_label':fs, 'cbar_tick_label':fs}

    sb_cmap = sns.color_palette(cmap, as_cmap=True)
    
    fig = plt.figure(figsize=(30,12))

    im = healpy.newvisufunc.projview(m=gal_hpx_map, graticule=True, graticule_labels=False, 
                            rot=(0,-90), rot_graticule=False, coord='G', projection_type="mollweide",
                            cmap=sb_cmap, min=vmin, max=vmax, width=30,
                            xsize=2000, cbar=False, fig=fig, badcolor='lightgrey',
                            fontsize=fs_dict,override_plot_properties={'figure_size_ratio':0.55})

    plt.title(title, fontsize=fs+6)

    cbar = plt.colorbar(im,orientation='horizontal', fraction=0.05, pad=0.02)
    cbar.set_label(cbarlabel, fontsize=fs)
    cbar.ax.tick_params(labelsize=fs, colors='k',grid_color='k')
    cbar.set_ticks(cbar_ticks)

    for spine in cbar.ax.spines.values():
            spine.set_visible(True)
            spine.set_linewidth(2)
    
    plt.savefig('/home/aordog/Dropbox/gmims_signs/'+outname+'.pdf')
    plt.savefig('/home/aordog/Dropbox/gmims_signs/'+outname+'.png')
    plt.close();

    return

In [ ]:
make_mollweide_map(map_new_q_hpx, 
                   -0.3, 0.3, 
                   np.arange(-0.3,0.4,0.1),
                   cmap='RdBu_r', 
                   outname='STAPS_Q_1373MHz_DRAO_export_galpol',
                   title='STAPS Stokes $Q$ at 1373 MHz, rotated to Galactic using DRAO export',
                   cbarlabel=r'Stokes $Q$ (K)')

In [ ]:
make_mollweide_map(map_new_u_hpx, 
                   -0.3, 0.3, 
                   np.arange(-0.3,0.4,0.1),
                   cmap='RdBu_r', 
                   outname='STAPS_U_1373MHz_DRAO_export_galpol',
                   title='STAPS Stokes $U$ at 1373 MHz, rotated to Galactic using DRAO export',
                   cbarlabel=r'Stokes $U$ (K)')

In [ ]:
make_mollweide_map(0.5*np.arctan2(map_new_u_hpx, map_new_q_hpx)*180/np.pi,
                   -90,90, 
                   np.arange(-90,120,30),
                   cmap='twilight_shifted', 
                   outname='STAPS_PA_1373MHz_DRAO_export_galpol',
                   title='STAPS PA at 1373 MHz, rotated to Galactic using DRAO export',
                   cbarlabel=r'PA (deg)')